# TFTokenizer example

Fit TFTokenizer on chr19 (hg38, JASPAR 2026 motif hits)

In [ ]:
from pathlib import Path

TRAIN, CHROMS = ["chr19"], ["chr19", "chr22"]
HITS, OUT = "data/hits/{chrom}.tsv.gz", Path("tokens")
JASPAR = "https://hgdownload.soe.ucsc.edu/gbdb/hg38/jaspar/JASPAR2026.bb"

## 1. Download

In [ ]:
import subprocess

for chrom in CHROMS:
    hits = Path(HITS.format(chrom=chrom))
    if not hits.exists():
        hits.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(f"bigBedToBed -chrom={chrom} -tsv {JASPAR} /dev/stdout | gzip > {hits}", shell=True, check=True)

## 2. Fit and tokenize


In [ ]:
import time
from tftokenizer import TFTokenizer

t = time.time()
tok = TFTokenizer().fit([(HITS.format(chrom=c), c) for c in TRAIN])
OUT.mkdir(exist_ok=True)
tok.save(OUT / "model.npz")
print(f"fit on {TRAIN}: {time.time() - t:.0f} s, {len(tok.motifs)} motifs")
for chrom in CHROMS:
    t = time.time(); tok.tokenize(HITS.format(chrom=chrom), chrom, OUT / chrom)
    print(f"tokenize {chrom}: {time.time() - t:.0f} s")

## 3. Output


In [ ]:
import numpy as np, zarr

for chrom in CHROMS:
    bins, tokens = np.load(OUT / chrom / "bins.npy"), zarr.open_array(str(OUT / chrom / "tokens.zarr"), mode="r")
    print(f"{chrom}: {tokens.shape[0]:,} tokens × {tokens.shape[1]} motifs, median length {np.median(np.diff(bins)):.0f} bp, "
          f"{1000 * tokens.shape[0] / bins[-1]:.1f} tokens / kb")